# Comparación medida: YOLO26n, YOLO11n y YOLOv8n
Detección sobre el mismo piloto real PlantDoc de dos enfermedades.
No se usan métricas de ejemplo ni se declara un ganador por adelantado.
No se entrenan Faster R-CNN ni MobileNet en este cuaderno.
El experimento de insectos con SAHI requiere datos de insectos revisados
y queda separado del piloto actual de hojas completas.

## 1. Proyecto y dependencias
En Colab, selecciona GPU para entrenar. La raíz debe contener `scripts/`,
`requirements-vision.txt` y los cuadernos, además de los datasets.
Puedes definir `IOTTESIS_ROOT` antes de esta celda para usar otra ubicación.
La instalación no cambia deliberadamente la versión CUDA/PyTorch de Colab.

In [ ]:
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None:
    drive.mount('/content/drive')
    os.environ.setdefault('IOTTESIS_ROOT', '/content/drive/MyDrive/Tesis_IOT_Papa')

candidatas = ([Path(os.environ['IOTTESIS_ROOT'])] if os.environ.get('IOTTESIS_ROOT') else [])
candidatas += [Path.cwd(), *Path.cwd().parents, Path('/content/IOTtesis')]
PROJECT_ROOT = next((p.resolve() for p in candidatas
                     if (p / 'scripts/entrenar_yolo26.py').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Sube/clona el proyecto actualizado y configura IOTTESIS_ROOT con su carpeta raíz.')
sys.path.insert(0, str(PROJECT_ROOT))
print('Proyecto:', PROJECT_ROOT)


In [ ]:
%pip install -q -r "{PROJECT_ROOT / 'requirements-vision.txt'}"


In [ ]:
import json
import torch
import ultralytics
from scripts.entrenar_yolo26 import training_plan, train_experiment, evaluate_test, benchmark_cpu

print('Ultralytics:', ultralytics.__version__)
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if tuple(int(v) for v in ultralytics.__version__.split('.')[:2]) < (8, 4):
    raise RuntimeError('Reinicia la sesión después de instalar Ultralytics compatible con YOLO26.')
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu':
    print('Sin GPU: los entrenamientos completos pueden ser lentos; no son necesarios para verificar el plan.')


## 2. Verificar el dataset y fijar el experimento
Ejecuta antes la preparación de `02` o `05`. Las tres familias reciben
las mismas particiones, épocas, tamaño de imagen, lote, semilla,
optimizador SGD y scheduler cosine. Conservar también las versiones
registradas: cambiar de entorno puede cambiar los resultados.

In [ ]:
FAMILIES = ['yolo26', 'yolo11', 'yolov8']
EPOCHS = 50
planes = [training_plan(PROJECT_ROOT, 'plantdoc', family, epochs=EPOCHS, batch=8)
          for family in FAMILIES]
print([(p['family'], p['initial_weights'], p['images']) for p in planes])


## 3. Entrenar y evaluar validación
Cada llamada inicia un entrenamiento. No utiliza test para seleccionar
hiperparámetros. El checkpoint se obtiene del directorio real de cada ejecución.

In [ ]:
experimentos = [train_experiment(plan, device=DEVICE) for plan in planes]
import pandas as pd
tabla_val = pd.DataFrame([{'Familia': r['family'], **r['validation']} for r in experimentos])
display(tabla_val)


## 4. Congelar selección mediante validación
Criterio explícito: mayor mAP50-95 de validación. No se supone que YOLO26
será superior; la utilidad para papa se determina con este experimento.

In [ ]:
metrica_seleccion = 'metrics/mAP50-95(B)'
ganador = max(experimentos, key=lambda r: r['validation'][metrica_seleccion])
print('Selección por validación:', ganador['family'], ganador['best_weights'])
resultados_dir = PROJECT_ROOT / 'runs/datasets_reales'
(resultados_dir / 'seleccion_por_validacion.json').write_text(json.dumps({
    'criterio': metrica_seleccion, 'family': ganador['family'],
    'best_weights': ganador['best_weights'], 'validation': ganador['validation']
}, indent=2))


## 5. Evaluación final y latencia CPU medida
Evaluar test después de congelar la comparación. No usar estos resultados
para volver a seleccionar el ganador ni ajustar parámetros. El benchmark
usa las mismas primeras imágenes de validación y mide la llamada predict
completa en CPU, con calentamiento; no equivale a latencia extremo a extremo
del sistema IoT ni a un benchmark oficial de Ultralytics.

In [ ]:
experimentos_finales = [evaluate_test(r, device=DEVICE) for r in experimentos]
filas = []
for record in experimentos_finales:
    benchmark = benchmark_cpu(record, samples=10)
    filas.append({'Familia': record['family'], **record['test'],
                  'Latencia CPU medida [ms]': benchmark['mean_ms'],
                  'Muestras benchmark': benchmark['samples']})
tabla = pd.DataFrame(filas)
display(tabla)
tabla.to_csv(resultados_dir / 'comparativa_yolo_test_medida.csv', index=False)


## 6. Exportación opcional del modelo elegido por validación
Los modelos .pt ya sirven para el backend. ONNX requiere sus dependencias
específicas y el formato de salida depende de la versión/cabeza exportada.
No asumir el layout de YOLO11 para YOLO26 en un consumidor ONNX propio.

In [ ]:
EXPORTAR_ONNX = False
if EXPORTAR_ONNX:
    from ultralytics import YOLO
    # Instalar onnx/onnxruntime antes si no están disponibles.
    ruta_onnx = YOLO(ganador['best_weights']).export(format='onnx', imgsz=640, device='cpu')
    print('ONNX:', ruta_onnx)
